# AgentCore Runtime with LangChain and LangGraph

Third framework, same four commands. If notebooks 1 and 2 landed, you already know everything on the platform side. The only thing that changes in this notebook is the contents of `main.py`.

| | NB1 Strands | NB2 no framework | NB3 LangGraph |
|---|---|---|---|
| Model access | `BedrockModel` | `boto3.converse` by hand | `ChatBedrockConverse` |
| Agent loop | framework | ~40 lines you own | compiled graph |
| Tools | `@tool` | `toolSpec` JSON | `@tool` |
| Session history | none (stateless per call) | none | checkpointer keyed on `thread_id` |
| Everything else | identical | identical | identical |

All three call the **same** Bedrock Converse API underneath. Notebook 2 wrote the loop by hand, so you have already seen the bottom of the stack that these two frameworks sit on.

```mermaid
flowchart TD
    S[Strands Agent] --> CV[Bedrock Converse API]
    H[hand written loop] --> CV
    L[LangGraph create_agent] --> CV
    CV --> M[Claude Haiku 4.5]
```

## 1. A third project, for the same reason as the second

| Option | Consequence |
|---|---|
| Third project (this notebook) | Its own CloudFormation stack. A LangGraph dependency failure cannot break the Strands demo you already got working |
| `agentcore add agent --name LangAgent --framework LangChain_LangGraph` | One stack for all three. One `deploy`. One failure takes down all three |

For a live session, isolate. For a shipped product where the agents version together, one project is the right call.

In [ ]:
import json, os, re, subprocess, uuid, pathlib
import boto3

REGION   = "us-east-1"
MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"

def sh(cmd, cwd=None, timeout=900, quiet=False):
    p = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True, timeout=timeout)
    out = ((p.stdout or "") + (p.stderr or "")).strip()
    if not quiet:
        print(f"$ {cmd}")
        print(out[:4000] if out else "(no output)")
        print("-" * 70)
    return p.returncode, out

WORKDIR = pathlib.Path(
    "/Users/akash-at-work/Documents/IBS Agentic AI and AWS GenAI Training/"
    "Day-11 - AgentCore/demos"
).expanduser()

PROJECT_NAME = "LangGraphRuntimeAgent"
AGENT_NAME   = "LangAgent"
PROJECT_DIR  = WORKDIR / PROJECT_NAME

print("workdir:", WORKDIR, "exists" if WORKDIR.exists() else "MISSING")
print("project:", PROJECT_DIR, "exists" if PROJECT_DIR.exists() else "not created yet")
sh("agentcore --version", quiet=True)

In [ ]:
if PROJECT_DIR.exists():
    print("project already exists, skipping create")
else:
    cmd = (
        "agentcore create "
        f"--project-name {PROJECT_NAME} "
        f"--name {AGENT_NAME} "
        "--type create --language Python --framework LangChain_LangGraph "
        "--model-provider Bedrock --protocol HTTP --build CodeZip --memory none"
    )
    sh(cmd, cwd=str(WORKDIR), timeout=1800)

cfg_path = PROJECT_DIR / "agentcore" / "agentcore.json"
cfg = json.loads(cfg_path.read_text()) if cfg_path.exists() else {}
RT = (cfg.get("runtimes") or [{}])[0]
ENTRYPOINT = RT.get("entrypoint", "main.py")
AGENT_DIR  = (PROJECT_DIR / RT.get("codeLocation", f"app/{AGENT_NAME}")).resolve()
AGENT_FILE = AGENT_DIR / ENTRYPOINT.split(":")[0]

print("entrypoint    :", AGENT_FILE)
print("uvicorn module:", AGENT_FILE.stem + ":app")
print("runtimeVersion:", RT.get("runtimeVersion"))

tgt = PROJECT_DIR / "agentcore" / "aws-targets.json"
print("aws-targets   :", tgt.read_text().strip()[:300] if tgt.exists() else "missing")
print("\nIf aws-targets.json is an empty list, deploy fails with 'Target \"default\" not found'.")
print("Run `agentcore deploy` once interactively in a terminal to create it.")

## 2. Fact-check the scaffold before you teach from it

Read what the generator wrote. Three things in the LangChain scaffold need correcting, and two of them will break a live demo.

| What the scaffold does | Status | What we do instead |
|---|---|---|
| `from langgraph.prebuilt import create_react_agent` | **Deprecated since LangGraph v1.0.** The library raises `LangGraphDeprecatedSinceV10` and the message says to import `create_agent` from `langchain.agents` | `from langchain.agents import create_agent` |
| Wires a live public MCP endpoint into the tool list by default | **Demo killer.** If that host is unreachable, every invoke fails with an unhandled TaskGroup error and it looks like your agent is broken | Remove the MCP client for the first run. Add it back deliberately |
| Defaults to `global.anthropic.claude-sonnet-4-5-...` | The `global.` prefix is a real inference profile, roughly 10% cheaper than geographic routing with no residency guarantee. On an IAM-scoped lab account it throws AccessDenied | `us.anthropic.claude-haiku-4-5-20251001-v1:0` |

The scaffold also uses `ChatBedrock`. We use `ChatBedrockConverse`, which is built on the Converse API, the same one notebook 2 called directly, and the one with the better tool-calling path for Claude.

**The teaching point worth saying out loud:** generated code is a starting position, not an authority. Two of those three defaults come from AWS's own template.

## 3. The graph

`create_agent` compiles a small state machine. It is worth showing, because "agent" stops being a mystery once people see there are only two nodes and one conditional edge.

```mermaid
flowchart TD
    S([start]) --> MODEL[model node<br/>call the LLM with the message list]
    MODEL --> C{did it request<br/>a tool call?}
    C -- no --> E([end])
    C -- yes --> TOOLS[tools node<br/>run the function, append ToolMessage]
    TOOLS --> MODEL
```

Compare it against notebook 2's hand-written loop. Same shape. The difference is who maintains it, and what comes attached.

| Attached for free | What it saves you |
|---|---|
| Checkpointer | Conversation history per `thread_id`, no code |
| `ToolNode` | Argument parsing, error containment, parallel tool calls |
| `recursion_limit` (default 25 per invocation) | Your `MAX_TURNS` cap |
| Streaming event shapes | Token and step streaming without rebuilding the loop |
| Interrupts | Human approval gates mid-graph |

In [ ]:
AGENT_SRC = '''"""Minimum AgentCore Runtime agent: LangChain + LangGraph on Amazon Bedrock.

Contract: POST /invocations with {"prompt": "..."} -> {"result": "...", "session_id": "..."}
"""

import os

from bedrock_agentcore.runtime import BedrockAgentCoreApp
from langchain.agents import create_agent          # v1 import; langgraph.prebuilt is deprecated
from langchain.tools import tool
from langchain_aws import ChatBedrockConverse
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
REGION = os.environ.get("AWS_REGION", os.environ.get("AWS_DEFAULT_REGION", "us-east-1"))

SYSTEM_PROMPT = (
    "You are TravelMind, an airline support agent. "
    "Use get_pnr for any question about a booking. "
    "Answer in three sentences or fewer. Never invent a booking."
)

app = BedrockAgentCoreApp()
log = app.logger

# LangChain spans in CloudWatch GenAI Observability. Guarded so a local run
# without the instrumentation package still starts.
try:
    from opentelemetry.instrumentation.langchain import LangchainInstrumentor
    LangchainInstrumentor().instrument()
except Exception as exc:  # noqa: BLE001
    log.warning("LangChain OTEL instrumentation unavailable: %s", exc)

_BOOKINGS = {
    "JX48Q2": {"passenger": "Rao", "tier": "Gold",
               "segment": "BLR-DEL", "status": "CANCELLED"},
}


@tool
def get_pnr(pnr: str) -> dict:
    """Look up an airline booking by its PNR code."""
    return _BOOKINGS.get(str(pnr).strip().upper(), {"error": "PNR not found"})


# Built once at import. The checkpointer is process memory, shared across
# invocations, partitioned by thread_id. It is NOT durable storage.
model = ChatBedrockConverse(model_id=MODEL_ID, region_name=REGION,
                            max_tokens=512, temperature=0.2)
checkpointer = InMemorySaver()

graph = create_agent(
    model,
    tools=[get_pnr],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)


@app.entrypoint
def invoke(payload, context):
    prompt = payload.get("prompt") if isinstance(payload, dict) else None
    if not isinstance(prompt, str) or not prompt.strip():
        return {"error": "payload must contain a non-empty 'prompt' string"}

    session_id = getattr(context, "session_id", None) or "default-session"

    result = graph.invoke(
        {"messages": [HumanMessage(content=prompt)]},
        config={"configurable": {"thread_id": session_id}},
    )

    return {
        "result": result["messages"][-1].content,
        "session_id": session_id,
        "message_count": len(result["messages"]),
    }


if __name__ == "__main__":
    app.run()
'''

AGENT_DIR.mkdir(parents=True, exist_ok=True)
AGENT_FILE.write_text(AGENT_SRC)
print("wrote", AGENT_FILE, f"({len(AGENT_SRC)} bytes)")
print("\nThe scaffold's model/ and mcp_client/ folders are now unused by this entrypoint.")

### Read the file

| Line | Why |
|---|---|
| `create_agent(...)` from `langchain.agents` | The supported v1 entry point. `create_react_agent` still runs and still emits a deprecation warning |
| `system_prompt=` | `create_agent` renamed this. `create_react_agent` called it `prompt` |
| `ChatBedrockConverse` | Converse API, correct tool-calling path for Claude on Bedrock |
| `max_tokens` and `temperature` set, `top_p` absent | Claude 4.x treats `temperature` and `top_p` as mutually exclusive |
| `thread_id=session_id` | The single line that turns AgentCore's session id into LangGraph conversation memory |
| `graph` built at import | Compiling per request wastes time. The graph is stateless; the checkpointer holds the state |
| `message_count` in the response | Makes memory visible in the demo without printing the whole transcript |

In [ ]:
PYPROJECT_SRC = f'''[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "{AGENT_NAME.lower()}"
version = "0.1.0"
description = "AgentCore Runtime application using LangChain and LangGraph"
requires-python = ">=3.10"
dependencies = [
    "aws-opentelemetry-distro",
    "opentelemetry-instrumentation-langchain>=0.59.0",
    "bedrock-agentcore>=1.9.1",
    "botocore[crt]>=1.35.0",
    "langchain>=1.0.3",
    "langgraph>=1.0.2",
    "langchain-aws>=1.0.0",
]

[tool.hatch.build.targets.wheel]
packages = ["."]
'''

pyproject = AGENT_DIR / "pyproject.toml"
pyproject.write_text(PYPROJECT_SRC)
print(PYPROJECT_SRC)
print("Dropped from the scaffold: mcp, langchain-mcp-adapters. Add them the day you use a gateway.")

code, _ = sh("uv sync", cwd=str(AGENT_DIR), timeout=1200)
print("uv sync exit code:", code)
print("Non-zero means `agentcore dev` would still start a server on a broken venv.")

In [ ]:
# Verify the venv, then draw the graph from the compiled object itself.
venv_py = AGENT_DIR / ".venv" / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
print("venv python:", "found" if venv_py.exists() else "MISSING")

probe_src = f'''
import importlib.util, sys
sys.path.insert(0, {str(AGENT_DIR)!r})
for m in ["bedrock_agentcore", "langchain", "langgraph", "langchain_aws", "boto3"]:
    print(f"  {{m:<22}}", "OK" if importlib.util.find_spec(m) else "MISSING")

from main import graph
print()
print(graph.get_graph().draw_ascii())
'''
probe = AGENT_DIR / "_probe.py"
probe.write_text(probe_src.replace("from main import graph", f"from {AGENT_FILE.stem} import graph"))
if venv_py.exists():
    sh(f'"{venv_py}" "{probe}"', cwd=str(AGENT_DIR), timeout=600)
probe.unlink(missing_ok=True)

That diagram is generated from the object that is about to serve traffic, not drawn by hand. When someone asks "what is my agent actually doing", `graph.get_graph().draw_ascii()` and `draw_mermaid()` answer from the executing code. Paste the mermaid version into any doc and it stays honest as the graph changes.

In [ ]:
# In-process test. Two turns on ONE thread, then the same follow-up on a fresh thread.
harness = f'''
import json, sys
sys.path.insert(0, {str(AGENT_DIR)!r})
from {AGENT_FILE.stem} import invoke

class Ctx:
    def __init__(self, sid):
        self.session_id = sid

sid = "local-thread-A-" + "0" * 20

print("TURN 1 (thread A):")
print(json.dumps(invoke({{"prompt": "What is the status of PNR JX48Q2?"}}, Ctx(sid)), indent=2))

print("\\nTURN 2 (thread A, no PNR mentioned):")
print(json.dumps(invoke({{"prompt": "And who is the passenger on it?"}}, Ctx(sid)), indent=2))

print("\\nSAME QUESTION on a FRESH thread:")
print(json.dumps(invoke({{"prompt": "And who is the passenger on it?"}},
                        Ctx("local-thread-B-" + "0" * 20)), indent=2))

print("\\nGUARD:", invoke({{}}, Ctx(sid)))
'''
hp = AGENT_DIR / "_local_test.py"
hp.write_text(harness)
if venv_py.exists():
    sh(f'"{venv_py}" "{hp}"', cwd=str(AGENT_DIR), timeout=600)
hp.unlink(missing_ok=True)

Read the three outputs against each other:

| Call | Expected | What it proves |
|---|---|---|
| Turn 1, thread A | Answers from the tool, `message_count` around 4 | The graph loops through the tool node |
| Turn 2, thread A | Answers "Rao" with no PNR in the prompt, `message_count` grows | The checkpointer replayed the thread |
| Same question, thread B | Asks which booking, `message_count` back to a small number | Threads are isolated |

Notebook 1's Strands agent could not do turn 2, because it built a stateless Agent per call. That is the honest difference between the two files, and it costs one keyword argument.

## 4. What the checkpointer is, and what it is not

```mermaid
flowchart TD
    A[session id from the caller] --> B[thread_id in LangGraph]
    B --> C[InMemorySaver<br/>process memory]
    C --> D{process still alive?}
    D -- yes --> E[history replays]
    D -- no, cold start or new instance --> F[history is gone]
    F --> G[AgentCore Memory<br/>if you need durability]
```

| Property | InMemorySaver | AgentCore Memory |
|---|---|---|
| Survives a cold start | no | yes |
| Survives scale-out to a second instance | no | yes |
| Cross-session recall of preferences | no | yes, with a strategy |
| AWS resources to create and delete | none | one, plus a provisioning wait |
| Extraction latency | none | eventually consistent, seconds to a minute |

Good enough for a demo and for many single-instance workloads. Not a memory architecture. When someone in the room says "so we have memory now", this table is the answer.

## 5. Local server

Terminal A, from the project root:

```bash
cd "<path>/LangGraphRuntimeAgent"
agentcore dev -p 8082
```

Terminal B:

```bash
agentcore dev "What is the status of PNR JX48Q2?" -p 8082
curl http://localhost:8082/ping
```

Three projects, three ports: 8080, 8081, 8082. `agentcore invoke` is deployed-only. To hit a running dev server, pass the prompt to `agentcore dev`.

In [ ]:
# Deploy. Run `agentcore deploy` once interactively first if the target or CDK bootstrap is missing.
code, out = sh("agentcore deploy -y", cwd=str(PROJECT_DIR), timeout=3600)
print("deploy exit code:", code)

In [ ]:
code, status_out = sh("agentcore status --json", cwd=str(PROJECT_DIR), timeout=300, quiet=True)
arns = sorted(set(re.findall(r"arn:aws[\w-]*:bedrock-agentcore:[^\"'\s,]+runtime/[^\"'\s,]+", status_out)))
AGENT_ARN = arns[0] if arns else None
print("runtime ARN:", AGENT_ARN or "not deployed yet")
if not AGENT_ARN:
    print(status_out[:1200])

In [ ]:
# Session continuity against the DEPLOYED runtime, not the local process.
acr = boto3.client("bedrock-agentcore", region_name=REGION)

def ask(prompt: str, session_id: str, arn: str = None):
    resp = acr.invoke_agent_runtime(
        agentRuntimeArn=arn or AGENT_ARN,
        runtimeSessionId=session_id,
        payload=json.dumps({"prompt": prompt}).encode("utf-8"),
        qualifier="DEFAULT",
    )
    raw = b"".join(chunk for chunk in resp.get("response", []))
    text = raw.decode("utf-8", errors="replace").strip()
    if "data:" in text:
        return {"stream_events": [ln[5:].strip() for ln in text.splitlines()
                                  if ln.strip().startswith("data:")]}
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {"raw": text}

if AGENT_ARN:
    sid = "langgraph-demo-" + uuid.uuid4().hex      # 15 + 32 = 47 chars, minimum is 33
    print("TURN 1:", json.dumps(ask("What is the status of PNR JX48Q2?", sid), indent=2)[:700])
    print("\nTURN 2:", json.dumps(ask("And who is the passenger?", sid), indent=2)[:700])
    print("\nFRESH SESSION:",
          json.dumps(ask("And who is the passenger?", "fresh-" + uuid.uuid4().hex), indent=2)[:700])
else:
    print("Deploy first.")

If turn 2 in the cloud forgets what turn 1 knew, that is not a bug in your code. It means the second request landed on a different runtime instance, which is exactly what the checkpointer table predicts. Reproducing that in front of the room is a better lesson than any slide about durability.

In [ ]:
sh("agentcore logs --since 30m -n 40", cwd=str(PROJECT_DIR), timeout=300)

## 6. Where this fails, LangGraph edition

| Symptom | Cause | Fix |
|---|---|---|
| `LangGraphDeprecatedSinceV10` warning | The scaffold's `langgraph.prebuilt` import | `from langchain.agents import create_agent` |
| `create_agent() got an unexpected keyword argument 'prompt'` | Copied a `create_react_agent` call | The parameter is `system_prompt` |
| Every invoke fails with an unhandled TaskGroup error | The scaffold's default public MCP endpoint is unreachable | Empty the MCP client list for the first run |
| `AccessDeniedException` on the model | The scaffold's `global.` Sonnet default is not permitted by your policy | Switch to the `us.` Haiku profile |
| History leaks between users | Same `thread_id` for different callers | `thread_id` must be the session id, never a constant |
| Memory vanishes intermittently in the cloud | Cold start or a second instance | Expected. Move to AgentCore Memory for durability |
| `GraphRecursionError` | Tool loop never terminates, default limit is 25 per invocation | Fix the tool description first, raise the limit second |
| Deploy fails installing wheels | No `aarch64` wheel for `PYTHON_3_14` | Pin `runtimeVersion` to `PYTHON_3_12` in `agentcore.json` |
| Tool called with wrong arguments | Thin docstring | The docstring is the tool's specification, treat it as an API contract |

## 7. Cleanup, all three projects

```bash
for p in MyFirstRuntimeAgent PlainRuntimeAgent LangGraphRuntimeAgent; do
  cd "<demos path>/$p" && agentcore remove all && agentcore deploy -y
done
```

Removing the local config alone leaves the AWS resources running. The `deploy` after `remove` is what tears them down. Confirm in the CloudFormation console that all three stacks are gone.

## 8. Closing the arc

Three notebooks, one platform, three frameworks. Put the summary on screen and let the room draw the conclusion.

| Decision | Reversible in | Owned by |
|---|---|---|
| Which framework | one file, one afternoon | the team writing the agent |
| Streaming or single response | one function signature | the team writing the agent |
| Which model | one constant | product and finance |
| Which platform | IAM, networking, observability, cost, and a migration | architecture |

**The line worth ending on:** a framework argument that blocks a platform decision has the dependency backwards. Pick the platform on operations. Pick the framework on productivity. Then let the teams change the framework whenever they can justify the afternoon.

**Skeptic's question to leave open:** all three agents here answer one question with one tool. Nothing about that workload proves any framework choice. What workload would actually separate them? Multi-agent handoff, long-running human-in-the-loop approvals, and streaming partial results are where the three diverge, and none of them fits in a minimum deploy.